# 👁️ Track B Evaluator

This notebook scores the predictions of an in-car scene understanding (ISU) system. It does **not** run a model itself: run `participant_starter.ipynb` first, which implements `isu_system(image)`, runs it on every scene, and saves the predictions to `track_b/submissions/track_b_predictions.json`. This notebook then compares those predictions with the ground-truth label files.

## Metrics summary

- **Feature accuracy:** the fraction of evaluated feature values that match their ground-truth labels across all scenes (`correct_features / evaluated_features`).
- **Mean feature match rate:** the per-scene fraction of matched features (`matched_features / evaluated_features` for each scene), averaged over scenes. Unlike feature accuracy, every scene counts equally regardless of how many features it has.
- **Per-feature accuracy:** the accuracy of each individual feature across the scenes where it is evaluated, showing which features the system gets wrong most often.
- **Scene exact-match accuracy:** the fraction of scenes for which every evaluated feature is predicted correctly.
- **Latency:** total inference time and mean inference time per scene, as measured around `isu_system(image)` in the participant starter.

The report includes the per-feature match flags, per-scene matched-feature counts and pass/fail results, and per-scene inference times.

Run the cells from top to bottom.

## 1. ⚙️ Configure the evaluation

Install the Track B dependencies before running this notebook:

```bash
pip install -r track_b/requirements.txt
```

This setup cell locates the repository-level `data/labels/` folder (downloaded by the participant starter) and the prediction files in `track_b/submissions/`. Every `*.json` file in `submissions/` is scored as one submission.

[ISU-Test](https://github.com/ast-fortiss-tum/ISU-Test) can generate controllable synthetic in-car scenes for Track B development and training. Any evaluated scenes should use the competition feature definitions and labels, and submitted systems should remain reproducible.

In [1]:
import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
TRACK_B_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'track_b_evaluator.py').exists() else NOTEBOOK_DIR / 'track_b'
REPO_ROOT = TRACK_B_DIR.parent
if str(TRACK_B_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_B_DIR))

from track_b_evaluator import (
    DEFAULT_IGNORED_FEATURES,
    comparable_features,
    feature_metrics,
    normalize,
)

DATA_DIR = REPO_ROOT / 'data'
LABELS_DIR = DATA_DIR / 'labels'
SUBMISSIONS_DIR = TRACK_B_DIR / 'submissions'
OUTPUT_DIR = TRACK_B_DIR / 'track_b_evaluation_output'
IGNORED_FEATURES = DEFAULT_IGNORED_FEATURES.copy()

if not any(LABELS_DIR.glob('*.json')):
    raise FileNotFoundError(
        f'No label files found in {LABELS_DIR}. Run participant_starter.ipynb first; it downloads the dataset.'
    )
submission_paths = sorted(SUBMISSIONS_DIR.glob('*.json'))
if not submission_paths:
    raise FileNotFoundError(
        f'No prediction files found in {SUBMISSIONS_DIR}. Run participant_starter.ipynb first; '
        'it writes submissions/track_b_predictions.json.'
    )

print(f'Labels directory: {LABELS_DIR}')
print(f'Found {len(submission_paths)} submission(s): {[path.name for path in submission_paths]}')

FileNotFoundError: No prediction files found in C:\Users\levia\Documents\testing\ISU-Challenge\Starterkit-ICSE-2027\track_b\submissions. Run participant_starter.ipynb first; it writes submissions/track_b_predictions.json.

## 2. 📂 Load predictions and labels

This stage loads each prediction file and matches every predicted scene to its label file by scene stem (`sample_00000_sim` and `sample_00000` both map to `labels/sample_00000.json`). The expected feature values are always taken from the label files, not from the `expected` field stored in the prediction file, and each prediction is compared feature by feature.

In [ ]:
def load_labels(labels_dir):
    labels = {}
    for label_path in sorted(labels_dir.glob('*.json')):
        stem = label_path.stem[:-len('_label')] if label_path.stem.endswith('_label') else label_path.stem
        labels[stem] = json.loads(label_path.read_text())
    return labels


def base_stem(image_name):
    for suffix in ('_sim', '_real'):
        if image_name.endswith(suffix):
            return image_name[:-len(suffix)]
    return image_name


def score_submission(submission_path, labels):
    submission = json.loads(submission_path.read_text())
    rows, missing = [], []
    for row in submission['predictions']:
        stem = base_stem(row['image'])
        if stem not in labels:
            missing.append(row['image'])
            continue
        # Expected values come from the label files, not from the submission.
        expected = comparable_features(labels[stem], IGNORED_FEATURES)
        prediction = row.get('prediction') or {}
        feature_matches = {
            key: normalize(prediction.get(key)) == normalize(value)
            for key, value in expected.items()
        }
        matched = sum(feature_matches.values())
        rows.append({
            'image': row['image'],
            'expected': expected,
            'prediction': prediction,
            'feature_matches': feature_matches,
            'matched_features': matched,
            'evaluated_features': len(feature_matches),
            'feature_match_rate': matched / len(feature_matches) if feature_matches else None,
            'all_features_match': all(feature_matches.values()),
            'inference_seconds': row.get('inference_seconds'),
        })
    if missing:
        print(f'  WARNING: {len(missing)} prediction(s) without a label file were skipped: {missing[:5]}')
    return submission, rows


labels = load_labels(LABELS_DIR)
print(f'Loaded {len(labels)} label files')

scored = {}
for submission_path in submission_paths:
    print(f'Scoring {submission_path.name}')
    scored[submission_path.stem] = score_submission(submission_path, labels)
    submission, rows = scored[submission_path.stem]
    print(f"  {len(rows)} scenes scored; model: {submission.get('model', 'unknown')}")

## 3. 📊 Calculate metrics and save the report

This stage aggregates the per-scene results of each submission into feature-level accuracy, the number of correct and evaluated features, scene-level exact-match accuracy, and total and mean inference latency. It saves one report per submission to `track_b_evaluation_output/<submission>_report.json`.

In [2]:
def per_feature_accuracy(rows):
    """Accuracy of each feature across all scenes where it is evaluated."""
    counts = {}
    for row in rows:
        for key, match in row['feature_matches'].items():
            correct, evaluated = counts.get(key, (0, 0))
            counts[key] = (correct + match, evaluated + 1)
    return {
        key: {'accuracy': correct / evaluated, 'correct': correct, 'evaluated': evaluated}
        for key, (correct, evaluated) in sorted(counts.items())
    }


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
reports = {}

for name, (submission, rows) in scored.items():
    times = [row['inference_seconds'] for row in rows if row['inference_seconds'] is not None]
    scene_rates = [row['feature_match_rate'] for row in rows if row['feature_match_rate'] is not None]
    metrics = {
        **feature_metrics(rows),
        'feature_match_rate_mean': sum(scene_rates) / len(scene_rates) if scene_rates else None,
        'scene_exact_match': (
            sum(row['all_features_match'] for row in rows) / len(rows) if rows else None
        ),
        'latency_seconds_total': sum(times) if times else None,
        'latency_seconds_mean': sum(times) / len(times) if times else None,
        'scenes': len(rows),
        'per_feature_accuracy': per_feature_accuracy(rows),
    }
    report = {
        'track': 'B',
        'submission': name,
        'model': submission.get('model'),
        'device': submission.get('device'),
        'features': submission.get('features'),
        'metrics': metrics,
        'predictions': rows,
    }
    output_path = OUTPUT_DIR / f'{name}_report.json'
    output_path.write_text(json.dumps(report, indent=2, ensure_ascii=True))
    reports[name] = report
    print(f'\n{name}')
    summary = {key: value for key, value in metrics.items() if key not in ('unique_values', 'per_feature_accuracy')}
    print(json.dumps(summary, indent=2))
    print(f'Saved report to {output_path}')

NameError: name 'scored' is not defined

In [ ]:
import pandas as pd

for name, report in reports.items():
    print(f'\n{name}')
    metrics = report['metrics']
    display(pd.Series(
        {key: value for key, value in metrics.items() if key not in ('unique_values', 'per_feature_accuracy')},
        name='value',
    ).to_frame())

    print('Accuracy per feature (lowest first):')
    feature_df = pd.DataFrame.from_dict(metrics['per_feature_accuracy'], orient='index')
    display(feature_df.sort_values('accuracy'))

    print('Unique ground-truth values per feature:')
    print(json.dumps(metrics['unique_values'], indent=2))

    print('Per-scene results:')
    results_df = pd.DataFrame([
        {
            'scene': row['image'],
            'matched_features': row['matched_features'],
            'evaluated_features': row['evaluated_features'],
            'feature_match_rate': row['feature_match_rate'],
            'all_features_match': row['all_features_match'],
            'inference_seconds': row['inference_seconds'],
        }
        for row in report['predictions']
    ])
    display(results_df.head())